In [1]:
print ("Group Project II")

Group Project II


In [2]:
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
/Users/zikrazuhuree/Documents/project-2-group-buy-now-pay-later-industry-project-21/notebooks


In [4]:
from pathlib import Path

# Notebook is inside /notebooks, so the project root is one folder above
PROJECT_ROOT = Path.cwd().parent
TABLES = PROJECT_ROOT / "data" / "tables"

print("Project root:", PROJECT_ROOT)
print("Tables folder:", TABLES)
print("Tables folder exists:", TABLES.exists())

Project root: /Users/zikrazuhuree/Documents/project-2-group-buy-now-pay-later-industry-project-21
Tables folder: /Users/zikrazuhuree/Documents/project-2-group-buy-now-pay-later-industry-project-21/data/tables
Tables folder exists: True


In [5]:
for item in sorted(TABLES.iterdir()):
    print(item.name)

consumer_fraud_probability.csv
consumer_user_details.parquet
merchant_fraud_probability.csv
tbl_consumer.csv
tbl_merchants.parquet
transactions_20210228_20210827_snapshot
transactions_20210828_20220227_snapshot
transactions_20220228_20220828_snapshot


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("BNPL_Preprocessing")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/11 08:26:32 WARN Utils: Your hostname, zikras-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 100.94.178.193 instead (on interface en0)
26/09/11 08:26:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 08:26:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.4


In [7]:
consumer = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .option("inferSchema", True)
    .csv(str(TABLES / "tbl_consumer.csv"))
)

user_details = spark.read.parquet(
    str(TABLES / "consumer_user_details.parquet")
)

merchants = spark.read.parquet(
    str(TABLES / "tbl_merchants.parquet")
)

consumer_fraud = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(TABLES / "consumer_fraud_probability.csv"))
)

merchant_fraud = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(TABLES / "merchant_fraud_probability.csv"))
)

In [8]:
# what speak thinks of the datasets
datasets = {
    "consumer": consumer,
    "user_details": user_details,
    "merchants": merchants,
    "consumer_fraud": consumer_fraud,
    "merchant_fraud": merchant_fraud
}

for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name.upper())
    print("=" * 60)
    df.printSchema()


CONSUMER
root
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postcode: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- consumer_id: integer (nullable = true)


USER_DETAILS
root
 |-- user_id: long (nullable = true)
 |-- consumer_id: long (nullable = true)


MERCHANTS
root
 |-- name: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- merchant_abn: long (nullable = true)


CONSUMER_FRAUD
root
 |-- user_id: integer (nullable = true)
 |-- order_datetime: date (nullable = true)
 |-- fraud_probability: double (nullable = true)


MERCHANT_FRAUD
root
 |-- merchant_abn: long (nullable = true)
 |-- order_datetime: date (nullable = true)
 |-- fraud_probability: double (nullable = true)



In [9]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)


CONSUMER
[('name', 'string'), ('address', 'string'), ('state', 'string'), ('postcode', 'int'), ('gender', 'string'), ('consumer_id', 'int')]

USER_DETAILS
[('user_id', 'bigint'), ('consumer_id', 'bigint')]

MERCHANTS
[('name', 'string'), ('tags', 'string'), ('merchant_abn', 'bigint')]

CONSUMER_FRAUD
[('user_id', 'int'), ('order_datetime', 'date'), ('fraud_probability', 'double')]

MERCHANT_FRAUD
[('merchant_abn', 'bigint'), ('order_datetime', 'date'), ('fraud_probability', 'double')]


In [12]:
for name, df in {
    "tx_1": tx_1,
    "tx_2": tx_2,
    "tx_3": tx_3
}.items():

    print(f"\n{name.upper()}")
    print(df.dtypes)


TX_1
[('user_id', 'bigint'), ('merchant_abn', 'bigint'), ('dollar_value', 'double'), ('order_id', 'string'), ('order_datetime', 'date')]

TX_2
[('user_id', 'bigint'), ('merchant_abn', 'bigint'), ('dollar_value', 'double'), ('order_id', 'string'), ('order_datetime', 'date')]

TX_3
[('user_id', 'bigint'), ('merchant_abn', 'bigint'), ('dollar_value', 'double'), ('order_id', 'string'), ('order_datetime', 'date')]


26/09/11 13:57:03 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 3928099 ms exceeds timeout 120000 ms
26/09/11 13:57:03 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/11 13:57:05 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at 

In [11]:
tx_1 = spark.read.parquet(
    str(TABLES / "transactions_20210228_20210827_snapshot")
)

tx_2 = spark.read.parquet(
    str(TABLES / "transactions_20210828_20220227_snapshot")
)

tx_3 = spark.read.parquet(
    str(TABLES / "transactions_20220228_20220828_snapshot")
)

print("Transaction snapshots loaded successfully.")

Transaction snapshots loaded successfully.
